In [1]:
pip install torch transformers datasets peft accelerate

Import Libraries

In [29]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

Prepare Your Dataset

In [30]:
data = [
    {"text": "Quantum cryptography allows secure communication using qubits."},
    {"text": "RSA encryption relies on large prime numbers and modular arithmetic."},
    {"text": "Elliptic curve cryptography provides smaller key sizes for the same security."},
    {"text": "Symmetric encryption uses the same key for encryption and decryption."},
]

dataset = Dataset.from_list(data)

Load GPT2 and tokenizer

In [31]:
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # GPT2 needs a pad token

model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
# List GPT2 model modules for LoRA
for name, module in model.named_modules():
    if "attn" in name and isinstance(module, torch.nn.Linear):
        print(name)

Tokenize Dataset

In [46]:
def tokenize(example):
    tokenized_inputs = tokenizer(example["text"], padding="max_length", truncation=True, max_length=64)
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].copy()
    return tokenized_inputs

dataset = dataset.map(tokenize)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Apply LoRA for Efficient Fine-Tuning

In [34]:
 lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn"],  # GPT2 attention linear layers
)

model = get_peft_model(model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Training Setup

In [48]:
from torch.utils.data import DataLoader

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Create DataLoader
train_loader = DataLoader(dataset, batch_size=2, shuffle=True)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

# Training loop
epochs = 3
model.train()

for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    for batch in train_loader:
        # Move batch to device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # For causal LM, labels = input_ids
        labels = input_ids.clone()

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Loss: {loss.item():.4f}")

Epoch 1/3


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Loss: 8.0626
Loss: 8.1741
Epoch 2/3
Loss: 8.1734
Loss: 8.2933
Epoch 3/3
Loss: 7.5757
Loss: 7.6749


In [49]:
prompt = "Modern cryptography techniques include"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    outputs = model.generate(**inputs, max_length=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Modern cryptography techniques include:

The use of a cryptographic hash function to generate a hash of a cryptographic key.

The use of a cryptographic hash function to generate a hash of a cryptographic key. The use of a cryptographic hash function to generate


Save Your Fine-Tuned Model

In [50]:
# Save LoRA-adapted model
model.save_pretrained("./llm_lora_crypto_model")
tokenizer.save_pretrained("./llm_lora_crypto_model")

('./llm_lora_crypto_model/tokenizer_config.json',
 './llm_lora_crypto_model/tokenizer.json')

In [51]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("./llm_lora_crypto_model")
tokenizer = AutoTokenizer.from_pretrained("./llm_lora_crypto_model")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights: 0it [00:00, ?it/s]

GPT2LMHeadModel LOAD REPORT from: ./llm_lora_crypto_model
Key                                                                       | Status     | 
--------------------------------------------------------------------------+------------+-
base_model.model.transformer.h.{0...11}.attn.c_attn.lora_A.default.weight | UNEXPECTED | 
base_model.model.transformer.h.{0...11}.attn.c_attn.lora_B.default.weight | UNEXPECTED | 
transformer.h.{0...11}.attn.c_attn.lora_B.default.weight                  | MISSING    | 
transformer.h.{0...11}.attn.c_attn.lora_A.default.weight                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Test Text Generation

In [52]:
prompt = "Advanced cryptography methods include"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    outputs = model.generate(**inputs, max_length=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Advanced cryptography methods include:

The use of a cryptographic key to encrypt the data.

The use of a cryptographic key to encrypt the data. The use of a cryptographic key to encrypt the data. The use of a cryptographic key to encrypt


Evaluate Model

In [53]:
model.eval()
total_loss = 0
with torch.no_grad():
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = input_ids.clone()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_loss += outputs.loss.item()

avg_loss = total_loss / len(train_loader)
print(f"Average training loss: {avg_loss:.4f}")

Average training loss: 8.9118
